## Camada Bronze —> Ingestão de Dados Brutos

Este notebook realiza a ingestão dos arquivos CSV provenientes do sistema legado da empresa do desafio.
Os dados são lidos sem transformações e mantidos no formato Delta Table, preservando a fidelidade dos dados originais.
Esta camada serve como fonte de verdade para as etapas subsequentes do pipeline.

**Fonte:** `/Volumes/dados/default/desafio_03/Data/`  
**Destino:** `/Volumes/dados/default/desafio_03/bronze/`  
**Tabelas:** clientes, produtos, vendedores, pedidos, itens_pedido

In [0]:
# 01_bronze.py
# Camada Bronze — Importação dos CSVs para as tabelas

# Caminho dos arquivos fontes
VOLUME_PATH = "/Volumes/dados/default/desafio_03/Data"
BRONZE_PATH = "/Volumes/dados/default/desafio_03/bronze"

# 1. Leitura dos CSV

clientes = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{VOLUME_PATH}/clientes.csv")

produtos = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{VOLUME_PATH}/produtos.csv")

vendedores = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{VOLUME_PATH}/vendedores.csv")

pedidos = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{VOLUME_PATH}/pedidos.csv")

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema_itens = StructType([
    StructField("id_item",        IntegerType(), True),
    StructField("id_pedido",      IntegerType(), True),
    StructField("id_produto",     IntegerType(), True),
    StructField("quantidade",     IntegerType(), True),
    StructField("marca",          StringType(),  True),
    StructField("preco_unitario", DoubleType(),  True),
])

itens_pedido = spark.read.format("csv") \
    .option("header", "true") \
    .schema(schema_itens) \
    .load(f"{VOLUME_PATH}/itens_pedido.csv")

# 2. Salvando como Delta Tables na camada Bronze 

clientes.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/clientes")
produtos.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/produtos")
vendedores.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/vendedores")
pedidos.write.format("delta").mode("overwrite").save(f"{BRONZE_PATH}/pedidos")
itens_pedido.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{BRONZE_PATH}/itens_pedido")

# 3. Registrando schemas

print("clientes")
clientes.printSchema()

print("produtos")
produtos.printSchema()

print("vendedores")
vendedores.printSchema()

print("pedidos")
pedidos.printSchema()

print("itens_pedido")
itens_pedido.printSchema()

print("Bronze concluído")